# Section 1: Why Traditional Patterns Break
## AI-Native Software Architecture — O'Reilly Course

We start with the simplest possible LLM architecture:

**One prompt → one model call → one output**

This is how many LLM applications begin. It is also why many of them fail in production.

In [12]:
import support_utils
from support_utils import (
    call_llm, customer_issues, primary_issue,
    USE_REAL_LLM, OPENAI_MODEL,
)
import json

print(f"Demo mode: USE_REAL_LLM={USE_REAL_LLM}")
print(f"Primary issue: {primary_issue}")

Demo mode: USE_REAL_LLM=True
Primary issue: I was charged twice for my subscription and need a refund.


In [13]:
# Toggle LLM mode without editing support_utils.py or restarting the kernel.
# Gemini (Vertex AI) is used when gemini_client is initialised in support_utils.
import support_utils
support_utils.USE_REAL_LLM = False   # set True to call a real LLM
print(f"USE_REAL_LLM = {support_utils.USE_REAL_LLM}")

USE_REAL_LLM = True


## Hands-On: Why Naive LLM Apps Break

### Task
Run the same naive LLM flow multiple times.

### Observe
- Does the output vary?
- Is the output structured?
- Could another service safely consume this output?

### Expected outcome
Understand why naive single-prompt systems are hard to trust in production.

In [14]:
def naive_support_assistant(issue: str) -> str:
    prompt = f"""Help the customer with this issue:

{issue}
"""
    return call_llm(prompt, temperature=0.9)

print("=== Naive outputs across repeated runs ===")
for i in range(5):
    print(f"\nRun {i + 1}:")
    print(naive_support_assistant(primary_issue))

=== Naive outputs across repeated runs ===

Run 1:


KeyboardInterrupt: 

## What did we observe?

We sent the same vague instruction multiple times and got outputs that may vary in:
- structure
- safety
- confidence
- downstream usability

This mirrors the production issue from the slides: LLM systems are probabilistic, context-sensitive,
and hard to reason about without architecture around them.

## Failure Mode Categorization

Before improving the system, let's categorize common failure modes from naive outputs.

In [15]:
failure_modes = [
    {"failure_mode": "Unstructured output",
     "why_it_matters": "Downstream services cannot reliably parse or validate it.",
     "example": "Freeform paragraph instead of JSON."},
    {"failure_mode": "Unsafe action",
     "why_it_matters": "The assistant may claim it processed a refund without authorization.",
     "example": "Sure, refund processed."},
    {"failure_mode": "Lack of grounding",
     "why_it_matters": "The model invents policy instead of using real company rules.",
     "example": "Refunds are always available."},
    {"failure_mode": "Domain confusion",
     "why_it_matters": "The assistant may answer unrelated questions instead of enforcing scope.",
     "example": "Explaining linked lists in a billing assistant."},
    {"failure_mode": "PII mishandling",
     "why_it_matters": "Sensitive data may be requested, logged, or repeated in unsafe ways.",
     "example": "Asking the user to send more personal information."},
]

for f in failure_modes:
    print(f"\n🔴 {f['failure_mode']}")
    print(f"Why it matters: {f['why_it_matters']}")
    print(f"Example: {f['example']}")


🔴 Unstructured output
Why it matters: Downstream services cannot reliably parse or validate it.
Example: Freeform paragraph instead of JSON.

🔴 Unsafe action
Why it matters: The assistant may claim it processed a refund without authorization.
Example: Sure, refund processed.

🔴 Lack of grounding
Why it matters: The model invents policy instead of using real company rules.
Example: Refunds are always available.

🔴 Domain confusion
Why it matters: The assistant may answer unrelated questions instead of enforcing scope.
Example: Explaining linked lists in a billing assistant.

🔴 PII mishandling
Why it matters: Sensitive data may be requested, logged, or repeated in unsafe ways.
Example: Asking the user to send more personal information.


## Section 1 takeaway

The naive approach produces:
- variable outputs on identical inputs
- unstructured responses that downstream systems cannot parse
- unsafe claims the system has no authority to make

**Next:** Section 2 — controlling behavior through input patterns.